This file is part of MOFTy.

MOFTy is free software: you can redistribute it and/or modify it under the
terms of the GNU General Public License version 3 as published by the Free
Software Foundation.

MOFTy is distributed in the hope that it will be useful, but WITHOUT ANY
WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR
A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with
MOFTy. If not, see http://www.gnu.org/licenses/

Copyright(C) 2026 Maximilian Neumann

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as stats
import seaborn as sns

The dataset is available from the 10x Genomics website:
https://www.10xgenomics.com/datasets/gene-and-protein-expression-library-of-human-glioblastoma-cytassist-ffpe-2-standard

We recommend using the batch download option. For this analysis, the following files are not required and may be excluded from the download:

* CytAssist_FFPE_Protein_Expression_Human_Glioblastoma_possorted_genome_bam.bam
* CytAssist_FFPE_Protein_Expression_Human_Glioblastoma_possorted_genome_bam.bam.bai
* CytAssist_FFPE_Protein_Expression_Human_Glioblastoma_molecule_info.h5
* CytAssist_FFPE_Protein_Expression_Human_Glioblastoma_cloupe.cloupe

Save the downloaded files to:

mofty/input/input_gbm/

Then extract all .tar, .tar.gz, and .zip archives in this directory.

In [ ]:
data_dir = Path("../input/input_gbm")

In [ ]:
h5_path = data_dir / "CytAssist_FFPE_Protein_Expression_Human_Glioblastoma_filtered_feature_bc_matrix.h5"

adata_full = sc.read_10x_h5(h5_path, gex_only=False)
isotype_mask = adata_full.var["isotype_control"].astype(str).str.upper().eq("TRUE")
normalized_mask = adata_full.var["normalized"].astype(str).str.upper().eq("TRUE")

# Determine modality column and add feat_modality
feat_col = "feature_types" if "feature_types" in adata_full.var.columns else "feature_type"
# print("Unique feature types:", adata_full.var[feat_col].unique())

def get_feat_modality(x):
    if x == "Gene Expression":
        return "gene_exp"
    elif x == "Antibody Capture":
        return "protein"
    else:
        return None

adata_full.var["feat_modality"] = adata_full.var[feat_col].map(get_feat_modality)

# Print duplicates by modality (before renaming)
gene_mask = adata_full.var["feat_modality"] == "gene_exp"
protein_mask = adata_full.var["feat_modality"] == "protein"

def _print_modality_dups(label, mask):
    names = pd.Index(adata_full.var_names[mask])
    dup_names = names[names.duplicated(keep=False)]
    if len(dup_names) == 0:
        print(f"No duplicate feature names in {label}.")
    else:
        print(f"Duplicate feature names in {label}:")
        print(pd.Series(dup_names).value_counts())

_print_modality_dups("Gene Expression", gene_mask)
_print_modality_dups("Antibody Capture", protein_mask)

# Print cross-modality duplicates (same name appears in both gene expression and protein)
names = pd.Index(adata_full.var_names)
# print(names)
cross_dups = names[names.duplicated(keep=False)]
if len(cross_dups) == 0:
    print("No cross-modality duplicate feature names.")
else:
    print("Cross-modality duplicate feature names and counts:")
    print(pd.Series(cross_dups).value_counts())

# If duplicated across modalities, suffix with modality
is_dup = names.duplicated(keep=False)
new_names = names.astype(str).tolist()
for i in range(len(new_names)):
    if is_dup[i]:
        new_names[i] = f"{new_names[i]}_{adata_full.var['feat_modality'].iloc[i]}"

# If duplicated within a modality, append _1, _2, ... (all copies get numbered)

keys = [(adata_full.var["feat_modality"].iloc[i], new_names[i]) for i in range(len(new_names))]
total_counts = Counter(keys)
seen_counts = Counter()
final_names = []
for i, n in enumerate(new_names):
    key = (adata_full.var["feat_modality"].iloc[i], n)
    seen_counts[key] += 1
    if total_counts[key] > 1:
        sec = adata_full.var["secondary_name"].iloc[i]
        new_n = f"{n}_{seen_counts[key]}"
        print(f"duplicate -> final: {new_n}, secondary_name: {sec}")
        final_names.append(new_n)
    else:
        final_names.append(n)

adata_full.var_names = final_names

# Load spatial metadata and attach
adata_vis = sc.read_visium(data_dir, count_file=h5_path.name)
adata_full.obsm["spatial"] = adata_vis.obsm["spatial"]
adata_full.uns["spatial"] = adata_vis.uns["spatial"]

adata = adata_full
adata.var["mt"] = adata.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

fig, axs = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(adata.obs["total_counts"], bins=50, kde=False, ax=axs[0])
sns.histplot(adata.obs["n_genes_by_counts"], bins=50, kde=False, ax=axs[1])
print(adata)

assert adata_full.var_names.is_unique, "Feature names are not unique after processing!"
print("Final feature name uniqueness check passed.")

In [ ]:
# ---------------------------------
# Parameters
# ---------------------------------
upper_gene_counts_quantile = 0.99
lower_gene_counts_quantile = 0.01
lower_genes_quantile = 0.01
min_gene_spots = 10
target_n_hvgs = 2000

lower_protein_counts_quantile = 0.01
lower_proteins_quantile = 0.01
min_protein_spots = 10


# Copy input
adata_filtered = adata.copy()
print(f"Original data shape: {adata_filtered.shape}")

# ---------------------------------
# Feature modality and initial masks
# ---------------------------------
if "feat_modality" not in adata_filtered.var.columns:
    raise KeyError("feat_modality missing in adata.var. Run the setup cell first.")

gene_mask = adata_filtered.var["feat_modality"] == "gene_exp"
protein_mask = adata_filtered.var["feat_modality"] == "protein"

# ---------------------------------
# 2) Gene expression spot QC
# ---------------------------------
adata_gene = adata_filtered[:, gene_mask].copy()
sc.pp.calculate_qc_metrics(adata_gene, inplace=True, percent_top=None)

min_counts = adata_gene.obs["total_counts"].quantile(lower_gene_counts_quantile)
max_counts = adata_gene.obs["total_counts"].quantile(upper_gene_counts_quantile)
min_genes = adata_gene.obs["n_genes_by_counts"].quantile(lower_genes_quantile)

print("Gene expression spot filtering thresholds:")
print(f"  min total_counts:      {min_counts:.2f}")
print(f"  max total_counts:      {max_counts:.2f}")
print(f"  min n_genes_by_counts: {min_genes:.2f}")

keep_obs_rna = (
    (adata_gene.obs["total_counts"] >= min_counts)
    & (adata_gene.obs["total_counts"] <= max_counts)
    & (adata_gene.obs["n_genes_by_counts"] >= min_genes)
)

adata_filtered = adata_filtered[keep_obs_rna].copy()
print(f"After gene expression spot filtering: {adata_filtered.shape}")

# Recompute masks
gene_mask = adata_filtered.var["feat_modality"] == "gene_exp"
protein_mask = adata_filtered.var["feat_modality"] == "protein"

# ---------------------------------
# 3) Identify isotype/control proteins
# ---------------------------------
if "isotype_control" in adata_filtered.var.columns:
    isotype_mask_meta = adata_filtered.var["isotype_control"].astype(str).str.upper().eq("TRUE")
else:
    isotype_mask_meta = pd.Series(False, index=adata_filtered.var_names)

isotype_mask = protein_mask & isotype_mask_meta
bio_protein_mask = protein_mask & ~isotype_mask

print(f"Total antibody features:   {protein_mask.sum()}")
print(f"Biological proteins:       {bio_protein_mask.sum()}")
print(f"Isotype/control features:  {isotype_mask.sum()}")

if isotype_mask.sum() > 0:
    print("Isotype/control features detected:")
    print(adata_filtered.var_names[isotype_mask].tolist())

# ---------------------------------
# 4) Protein-based spot QC (for spot filtering only)
#    Use biological proteins only
# ---------------------------------
adata_bio_protein_qc = adata_filtered[:, bio_protein_mask].copy()
sc.pp.calculate_qc_metrics(adata_bio_protein_qc, inplace=True, percent_top=None)

adata_bio_protein_qc.obs["total_protein_counts"] = adata_bio_protein_qc.obs["total_counts"]
adata_bio_protein_qc.obs["n_proteins_by_counts"] = adata_bio_protein_qc.obs["n_genes_by_counts"]

min_protein_counts = adata_bio_protein_qc.obs["total_protein_counts"].quantile(lower_protein_counts_quantile)
min_detected_proteins = adata_bio_protein_qc.obs["n_proteins_by_counts"].quantile(lower_proteins_quantile)

print("Protein spot filtering thresholds:")
print(f"  min total_protein_counts: {min_protein_counts:.2f}")
print(f"  min n_proteins_by_counts: {min_detected_proteins:.2f}")

keep_obs_protein = (
    (adata_bio_protein_qc.obs["total_protein_counts"] >= min_protein_counts)
    & (adata_bio_protein_qc.obs["n_proteins_by_counts"] >= min_detected_proteins)
)

adata_filtered = adata_filtered[keep_obs_protein].copy()
print(f"After protein-based spot filtering: {adata_filtered.shape}")

# Recompute masks
gene_mask = adata_filtered.var["feat_modality"] == "gene_exp"
protein_mask = adata_filtered.var["feat_modality"] == "protein"

if "isotype_control" in adata_filtered.var.columns:
    isotype_mask_meta = adata_filtered.var["isotype_control"].astype(str).str.upper().eq("TRUE")
else:
    isotype_mask_meta = pd.Series(False, index=adata_filtered.var_names)

isotype_mask = protein_mask & isotype_mask_meta
bio_protein_mask = protein_mask & ~isotype_mask

# ---------------------------------
# 5) Gene filtering (minimum spots)
# ---------------------------------
adata_gene = adata_filtered[:, gene_mask].copy()
sc.pp.calculate_qc_metrics(adata_gene, inplace=True, percent_top=None)

gene_keep = adata_gene.var["n_cells_by_counts"] >= min_gene_spots
keep_gene_names = adata_gene.var_names[gene_keep]

print(f"Keeping {gene_keep.sum()} / {adata_gene.shape[1]} genes with n_cells_by_counts >= {min_gene_spots}")

# ---------------------------------
# 6) Protein filtering (same logic as genes: minimum spots)
# ---------------------------------
adata_bio_protein_raw = adata_filtered[:, bio_protein_mask].copy()
sc.pp.calculate_qc_metrics(adata_bio_protein_raw, inplace=True, percent_top=None)

protein_keep = adata_bio_protein_raw.var["n_cells_by_counts"] >= min_protein_spots
keep_bio_protein_names = adata_bio_protein_raw.var_names[protein_keep].tolist()

print(
    f"Keeping {protein_keep.sum()} / {adata_bio_protein_raw.shape[1]} proteins "
    f"with n_cells_by_counts >= {min_protein_spots}"
)

# ---------------------------------
# 7) Subset to pre-HVG feature pool
#    (genes passing min spots + proteins passing min spots)
# ---------------------------------
var_keep = adata_filtered.var_names.isin(keep_gene_names) | adata_filtered.var_names.isin(keep_bio_protein_names)
adata_filtered = adata_filtered[:, var_keep].copy()
print(f"After gene/protein preselection: {adata_filtered.shape}")

# Recompute masks again
gene_mask = adata_filtered.var["feat_modality"] == "gene_exp"
protein_mask = adata_filtered.var["feat_modality"] == "protein"

# ---------------------------------
# 8) Split modalities
# ---------------------------------
adata_gene_exp = adata_filtered[:, gene_mask].copy()
adata_bio_protein = adata_filtered[:, protein_mask].copy()

# ---------------------------------
# 9) Normalize genes
# ---------------------------------
sc.pp.normalize_total(adata_gene_exp, inplace=True)
sc.pp.log1p(adata_gene_exp)

# ---------------------------------
# 10) CLR normalize proteins
# ---------------------------------
if adata_bio_protein.shape[1] > 0:
    X_prot = adata_bio_protein.to_df().to_numpy(dtype=np.float64)
    print("Protein matrix shape:", X_prot.shape)
    X_prot = X_prot + 1.0
    gmeans = stats.gmean(X_prot, axis=1)
    adata_bio_protein.X = np.log(X_prot / gmeans[:, None])

# ---------------------------------
# 11) Reassemble X
# ---------------------------------
X_full = np.zeros(adata_filtered.shape, dtype=np.float64)

X_gene = adata_gene_exp.to_df().to_numpy(dtype=np.float64)
X_full[:, gene_mask] = X_gene

X_bio = adata_bio_protein.to_df().to_numpy(dtype=np.float64)
X_full[:, protein_mask] = X_bio

adata_filtered.X = X_full

# ---------------------------------
# 12) HVGs on genes only: exactly target_n_hvgs
# ---------------------------------
if adata_gene_exp.n_vars < target_n_hvgs:
    raise ValueError(
        f"Only {adata_gene_exp.n_vars} genes available after filtering, but target_n_hvgs={target_n_hvgs}."
    )

sc.pp.highly_variable_genes(adata_gene_exp, flavor="seurat", n_top_genes=target_n_hvgs)

adata_filtered.var["highly_variable"] = False
adata_filtered.var.loc[gene_mask, "highly_variable"] = adata_gene_exp.var["highly_variable"].values

# ---------------------------------
# 13) FINAL SUBSET:
#     keep only HVGs + filtered proteins
# ---------------------------------
final_keep = adata_filtered.var["highly_variable"] | (adata_filtered.var["feat_modality"] == "protein")
adata_filtered = adata_filtered[:, final_keep].copy()

adata = adata_filtered

n_final_proteins = int((adata.var["feat_modality"] == "protein").sum())
n_final_hvgs = int(adata.var["highly_variable"].sum())
print(f"Final data shape: {adata.shape}")
print(f"Final proteins: {n_final_proteins}")
print(f"Final HVGs: {n_final_hvgs} (target={target_n_hvgs})")

if n_final_hvgs != target_n_hvgs:
    raise RuntimeError(f"Expected {target_n_hvgs} HVGs, got {n_final_hvgs}.")

In [ ]:
print(adata.X.shape)

In [ ]:
adata.obs = pd.concat(
    [
        adata.obs,
        pd.DataFrame(adata.obsm["spatial"], columns=["spatial1", "spatial2"], index=adata.obs_names),
    ],
    axis=1
)

In [ ]:
adata.write_h5ad(data_dir / "processed_adata.h5ad")

In [ ]:
gene_mask = adata.var["feat_modality"] == "gene_exp"
protein_mask = adata.var["feat_modality"] == "protein"
mdata = mu.MuData({
    "gene_exp": adata[:, gene_mask].copy(),
    "protein": adata[:, protein_mask].copy(),
})
mdata.mod["gene_exp"].varm.clear()
mdata.mod["gene_exp"] = mdata.mod["gene_exp"][:, mdata.mod["gene_exp"].var["highly_variable"]].copy()
# print("gene expression vars after HVG subset:", mdata.mod["gene_exp"].n_vars)
# print(mdata.mod["protein"].var_names)

mdata.obs[["spatial1", "spatial2"]] = adata.obsm["spatial"]
for view in ["gene_exp", "protein"]:
    sc.pp.calculate_qc_metrics(mdata.mod[view], inplace=True, percent_top=None)

# print(mdata)
mdata_output_file = data_dir / "processed_mdata.h5mu"
Path(mdata_output_file).unlink(missing_ok=True)
mdata.write(mdata_output_file)


In [ ]:
#Create untrained MEFISTO model (0 iterations)
mefisto_output_file = data_dir / "mefisto_model_untrained.hdf5"
Path(mefisto_output_file).unlink(missing_ok=True)
mdata_output_file = data_dir / "processed_mdata.h5mu"
mdata = mu.read_h5mu(mdata_output_file)
mu.tl.mofa(
    mdata,
    center_groups=False,
    likelihoods=["gaussian", "gaussian"],
    use_var=None,
    ard_factors=False,
    outfile=mefisto_output_file,
    smooth_kwargs={"sparseGP": True, "frac_inducing": 0.001}, # no practical impact, just to reduce memory usage
    smooth_covariate=["spatial1", "spatial2"],
    n_iterations=0,
    seed=42
)